# Civitas AI — Train Model A

## What you do before running
1. Runtime → Change runtime type → GPU (T4 on free, L4/A100 if Pro)
2. Upload `civitas_data_v2.zip` to MyDrive/civitas/
3. (Optional) Upload your repo zip or paste your GitHub URL below
4. Run all cells top-to-bottom
5. After tuning: run the final "TEST — RUN ONCE" cell exactly once

In [ ]:
import subprocess
result = subprocess.run(['nvidia-smi', '--query-gpu=name,memory.total', '--format=csv,noheader'], capture_output=True, text=True)
gpu_name = result.stdout.strip()
print(f"GPU: {gpu_name}")

if 'A100' in gpu_name or 'L4' in gpu_name:
    DTYPE = 'bf16'; BATCH_SIZE = 32; GRAD_ACCUM = 1
elif 'T4' in gpu_name or 'V100' in gpu_name:
    DTYPE = 'fp16'; BATCH_SIZE = 16; GRAD_ACCUM = 2  # T4 has no bf16
else:
    DTYPE = 'fp32'; BATCH_SIZE = 8; GRAD_ACCUM = 4
print(f"dtype={DTYPE}, batch={BATCH_SIZE}, accum={GRAD_ACCUM}")

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
PROJECT_DIR = '/content/drive/MyDrive/civitas'
import os; os.makedirs(PROJECT_DIR, exist_ok=True)
print(f"Project dir: {PROJECT_DIR}")

In [ ]:
# @param {type:"string"}
GITHUB_URL = ""  # paste your repo URL, or leave blank to use uploaded zip

import subprocess, os

subprocess.run(['pip', 'install', '-q', '-r', '/content/civitas/requirements.txt'] 
               if os.path.exists('/content/civitas') else
               ['pip', 'install', '-q', 'torch', 'transformers', 'datasets', 
                'pandas', 'scikit-learn', 'seqeval', 'accelerate', 'peft', 
                'onnx', 'onnxruntime', 'indic-transliteration', 'pyyaml'])

if GITHUB_URL:
    subprocess.run(['git', 'clone', GITHUB_URL, '/content/civitas'], check=True)
else:
    from google.colab import files
    print("Upload your repo zip:")
    uploaded = files.upload()
    fname = list(uploaded.keys())[0]
    subprocess.run(['unzip', '-q', fname, '-d', '/content/civitas'])

import sys; sys.path.insert(0, '/content/civitas')
print("Setup complete")

In [ ]:
import subprocess, json, shutil
DATA_ZIP = f"{PROJECT_DIR}/civitas_data_v2.zip"
DATA_DIR = '/content/data'
subprocess.run(['unzip', '-q', '-o', DATA_ZIP, '-d', DATA_DIR])

with open(f"{DATA_DIR}/manifest.json") as f:
    manifest = json.load(f)

print(f"Git commit: {manifest['git_commit']}")
print(f"Created: {manifest['created_at']}")
for split, info in manifest['splits'].items():
    print(f"  {split}: {info['rows']:,} rows")

In [ ]:
CONFIG = {
    'encoder_name': 'xlm-roberta-base',
    'max_length': 128,
    'epochs': 4,
    'lr': 2e-5,
    'warmup_ratio': 0.1,
    'weight_decay': 0.01,
    'batch_size': BATCH_SIZE,
    'grad_accum': GRAD_ACCUM,
    'dtype': DTYPE,
    'loss_weights': {'hate': 1.0, 'target': 0.5, 'severity': 0.5, 'rationale': 0.5},
    'hate_gamma': 0,      # 0 = plain cross-entropy (hate is only 57/43 imbalanced)
    'severity_gamma': 2,  # focal for severity (genuinely imbalanced)
    'script_augment_p': 0.3,
    'identity_aug': True,
    'checkpoint_every_n_steps': 500,
    'drive_checkpoint_dir': f'{PROJECT_DIR}/checkpoints/model_a',
    'data_dir': DATA_DIR,
}
import json; print(json.dumps(CONFIG, indent=2))

In [ ]:
import subprocess, os, json
import sys; sys.path.insert(0, '/content/civitas')

os.makedirs(CONFIG['drive_checkpoint_dir'], exist_ok=True)

# Check for existing checkpoint to resume from
latest_ckpt = None
ckpt_dir = CONFIG['drive_checkpoint_dir']
if os.path.exists(ckpt_dir):
    pts = sorted([f for f in os.listdir(ckpt_dir) if f.endswith('.pt')], 
                  key=lambda f: os.path.getmtime(os.path.join(ckpt_dir, f)))
    if pts:
        latest_ckpt = os.path.join(ckpt_dir, pts[-1])
        print(f"Resuming from: {latest_ckpt}")

cmd = [
    'python', '/content/civitas/train.py',
    '--data_dir', CONFIG['data_dir'],
    '--output_dir', ckpt_dir,
    '--encoder_name', CONFIG['encoder_name'],
    '--epochs', str(CONFIG['epochs']),
    '--batch_size', str(CONFIG['batch_size']),
    '--grad_accum', str(CONFIG['grad_accum']),
    '--lr', str(CONFIG['lr']),
    '--max_length', str(CONFIG['max_length']),
    '--script_augment_p', str(CONFIG['script_augment_p']),
]
if latest_ckpt:
    cmd += ['--resume', latest_ckpt]
if CONFIG.get('identity_aug'):
    cmd += ['--identity_aug', '1']

print('Command:', ' '.join(cmd))
result = subprocess.run(cmd, capture_output=False)
print('Exit code:', result.returncode)

## ⚠️ RUN ONCE AT THE VERY END
### Re-running this cell after further tuning invalidates the test set.
### Only run this when you are done with all tuning and ready to report final numbers.

In [ ]:
import subprocess, json, os
best_ckpt = os.path.join(CONFIG['drive_checkpoint_dir'], 'best_model.pt')
result = subprocess.run([
    'python', '/content/civitas/train.py',
    '--eval_only',
    '--checkpoint', best_ckpt,
    '--data_dir', CONFIG['data_dir'],
    '--output_dir', CONFIG['drive_checkpoint_dir'],
], capture_output=False)
print('Test evaluation exit code:', result.returncode)
report_path = os.path.join(CONFIG['drive_checkpoint_dir'], 'test_report.json')
if os.path.exists(report_path):
    with open(report_path) as f:
        print(json.dumps(json.load(f), indent=2))

In [ ]:
import subprocess
result = subprocess.run([
    'python', '/content/civitas/eval/eval_hatecheck.py',
    '--checkpoint', os.path.join(CONFIG['drive_checkpoint_dir'], 'best_model.pt'),
    '--device', 'cuda',
], capture_output=False)
print('HateCheck eval exit code:', result.returncode)